# 03. Cleaning and Standardising Tabular Data
**Engineering Data Processing with Python** | one-day course | about 40 minutes

> **TRAINER / SOLUTION VERSION.** Every blank is filled in and the notebook has been run.


In [1]:
# @title Setup: run this cell first (it loads the course data)
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## The point of this module

Take the Kilbrack work orders and asset register from "loads without error" to "safe to add up":
tidy text, fix category spellings, turn text into numbers, sort out dates and units, and decide what to do about
missing values. We keep an untouched copy of the raw data to compare against.

In [2]:
# @title Helpers for this notebook (just run it)
def parse_dates(series, formats):
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in formats:
        result = result.fillna(pd.to_datetime(series, format=fmt, errors="coerce"))
    return result

def to_kw(value):
    """'38 kW', '38kW', '38000 W' or 38 -> 38.0 (kW). Blank -> NaN."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).upper().replace(" ", "")
    if text.endswith("KW"):
        return float(text[:-2])
    if text.endswith("W"):
        return float(text[:-1]) / 1000
    return float(text)

def parse_install(value):
    """Handles real Excel dates plus text like 22/06/2019, 2016-01-01, 08.06.2014, March 2016."""
    if pd.isna(value):
        return pd.NaT
    if hasattr(value, "year"):
        return pd.Timestamp(value)
    for fmt in ["%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y", "%B %Y"]:
        try:
            return pd.to_datetime(str(value).strip(), format=fmt)
        except ValueError:
            pass
    return pd.NaT
print("Helpers ready: parse_dates(), to_kw(), parse_install()")

Helpers ready: parse_dates(), to_kw(), parse_install()


In [3]:
wo_raw = pd.read_csv("data/work_orders_cmms.csv", na_values=["-"])
wo = wo_raw.copy()          # we clean the copy, the raw stays untouched
print(wo.shape)
wo.head()

(1016, 12)


,wo_id,asset_id,wo_type,priority,date_raised,date_closed,status,labour_hours,parts_cost_eur,technician,description,last_modified
0,WO-25-00270,PMP-005,PM,3,23/06/2025 13:15,24/06/2025 01:17,Closed,2.5,222.81,Gráinne FitzGerald,Quarterly PM service,2025-06-24 03:41:14
1,WO-26-00692,PMP-005,pm,4,04/03/2026 18:45,06/03/2026 06:12,CLOSED,1.5,0.77,Orla Doyle,"Replace filters, check belts",2026-03-06 09:02:03
2,WO-25-00449,PMP-001,Inspection,3,10/10/2025 10:30,11/10/2025 08:37,Closed,1.5,€0.00,Ciarán Walsh,Vibration route survey,2025-10-11 09:35:23
3,WO-25-00466,PMP-009,PM,4,18/10/2025 07:00,18/10/2025 13:04,closed,1.0,€7.48,Patrick Ryan,Annual service per OEM schedule,2025-10-18 17:15:07
4,WO-25-00198,CNV-006,Corrective,4,06/05/2025 17:45,08/05/2025 19:56,Closed,1.5,€417.25,Niamh O Connor,"Belt snapped, replaced",2025-05-08 20:18:16


## 1. Tidy text

pandas text tools sit under `.str`. Example: the status column has `Closed`, `closed`, `CLOSED `. Remove spaces
and use Title Case:

In [4]:
print(wo["status"].value_counts(), "\n")
wo["status"] = wo["status"].str.strip().str.title()
print(wo["status"].value_counts())

status
Closed         570
closed         216
CLOSED         194
Open            30
In Progress      6
Name: count, dtype: int64 

status
Closed         980
Open            30
In Progress      6
Name: count, dtype: int64


### Your turn 3.1
Asset IDs come as `CNC-001`, `cnc-001` and `CNC-001 `. Remove spaces and make them **upper** case.
Choose from: `strip`, `upper`, `lower`, `title`.

In [5]:
print("Different asset IDs before:", wo["asset_id"].nunique())
wo["asset_id"] = wo["asset_id"].str.strip().str.upper()
print("Different asset IDs after: ", wo["asset_id"].nunique())

Different asset IDs before: 159
Different asset IDs after:  70


The count drops because `cnc-001` and `CNC-001 ` are now recognised as the same asset.

## 2. Fix category spellings

Twelve spellings of three work order types. Two steps: make them upper case so similar spellings collapse, then
**map** each spelling to an agreed code with a dictionary (a lookup table).

In [6]:
wo["wo_type"].str.strip().str.upper().value_counts()

wo_type
PM            505
CM            189
CORRECTIVE     98
PREVENTIVE     91
INSP           71
BREAKDOWN      42
INSPECTION     20
Name: count, dtype: int64

### Your turn 3.2
Complete the lookup table. `PREVENTIVE` is planned maintenance (`PM`). `BREAKDOWN` is corrective (`CM`).

In [7]:
TYPE_MAP = {
    "PM": "PM",
    "PREVENTIVE": "PM",
    "CM": "CM",
    "CORRECTIVE": "CM",
    "BREAKDOWN": "CM",
    "INSP": "INSP",
    "INSPECTION": "INSP",
}
wo["wo_type"] = wo["wo_type"].str.strip().str.upper().map(TYPE_MAP)
print("Spellings that did not map:", wo["wo_type"].isna().sum())
wo["wo_type"].value_counts()

Spellings that did not map: 0


wo_type
PM      596
CM      329
INSP     91
Name: count, dtype: int64

In [8]:
check(set(wo["wo_type"]) == {"PM", "CM", "INSP"}, "Three clean categories.", "Something did not map.")

PASS: Three clean categories.


A useful property of `.map()`: anything **not** in the lookup becomes blank. So when next month's export has a new
spelling, the "did not map" count tells you straight away instead of it slipping through.

## 3. Numbers stored as text

Example: parts cost arrives as `222.81`, `€1,250.00`, `N/A`. Remove `€` and the comma, then convert to numbers.
Then compare missing values **before and after**, to prove nothing was lost.

In [9]:
missing_before = wo["parts_cost_eur"].isna().sum()

cost_text = wo["parts_cost_eur"].str.replace("€", "").str.replace(",", "")
wo["parts_cost_eur"] = pd.to_numeric(cost_text, errors="coerce")

print("Missing before:", missing_before, " after:", wo["parts_cost_eur"].isna().sum())

Missing before: 92  after: 92


Same count before and after: nothing was lost in conversion. If "after" were bigger, some values failed to convert
and became blank. That before/after habit is the most useful thing in this module.

### Your turn 3.3
Convert `labour_hours` to numbers in the same way (no symbols to remove this time).

In [10]:
wo["labour_hours"] = pd.to_numeric(wo["labour_hours"], errors="coerce")
wo["labour_hours"].describe()

count    980.000000
mean       4.402551
std       14.907698
min       -4.000000
25%        2.000000
50%        3.500000
75%        4.500000
max      360.000000
Name: labour_hours, dtype: float64

Look at `min` and `max`: negative hours, and a job of 360 hours. Probably 360 *minutes* typed into an hours box.
We do not guess a fix. We flag it in Module 06.

Priority has `1`, `P1`, `Urgent` and blanks. The cleanup is written for you. Run and read.

In [11]:
p = wo["priority"].astype("string").str.strip().str.upper()
p = p.str.replace("P", "").str.replace("URGENT", "1")
wo["priority"] = pd.to_numeric(p, errors="coerce").astype("Int64")
wo["priority"].value_counts(dropna=False)

priority
3       463
4       411
2        86
1        35
<NA>     21
Name: count, dtype: Int64

## 4. Dates

Example: `date_raised`, with both formats in the file:

In [12]:
FORMATS = ["%d/%m/%Y %H:%M", "%Y-%m-%d %H:%M"]
wo["date_raised"] = parse_dates(wo["date_raised"], FORMATS)
wo["last_modified"] = parse_dates(wo["last_modified"], ["%Y-%m-%d %H:%M:%S"])

### Your turn 3.4
Do the same for `date_closed`.

In [13]:
wo["date_closed"] = parse_dates(wo["date_closed"], FORMATS)
print("Blank closed dates in the raw file:", wo_raw["date_closed"].isna().sum())
print("Blank closed dates now:            ", wo["date_closed"].isna().sum())

Blank closed dates in the raw file: 41
Blank closed dates now:             41


In [14]:
check(wo["date_closed"].isna().sum() == wo_raw["date_closed"].isna().sum(),
      "Every closed date parsed. The only blanks are jobs that were blank to begin with.",
      "Check you used the date_closed column.")

PASS: Every closed date parsed. The only blanks are jobs that were blank to begin with.


## 5. Missing values: decide, don't default

| Column | Blank means | Decision |
|---|---|---|
| `date_closed` | Job still open (or someone forgot) | Leave blank. Module 06 flags "Closed" jobs with no date |
| `parts_cost_eur` | Not recorded | Leave blank. Filling with the average would **invent money** |
| `labour_hours` | Not recorded | Leave blank and add a flag |

The common mistake is filling blanks with 0 or the average so the numbers "work". Then nobody can tell made-up values
from real ones.

In [15]:
wo["hours_missing"] = wo["labour_hours"].isna()
wo.isna().sum()

wo_id              0
asset_id           0
wo_type            0
priority          21
date_raised        0
date_closed       41
status             0
labour_hours      36
parts_cost_eur    92
technician         0
description        0
last_modified      0
hours_missing      0
dtype: int64

## 6. The asset register

Loading and column names are done for you (the column names become lower case with underscores).

In [16]:
reg = pd.read_excel("data/asset_register.xlsx", skiprows=3)
reg.columns = reg.columns.str.strip().str.lower().str.replace(" ", "_")
reg["asset_id"] = reg["asset_id"].str.strip().str.upper()
reg["site"].str.strip().str.title().value_counts()

site
Cork             20
Galway           17
Limerick         17
Cork Plant        6
Limerick Site     2
Name: count, dtype: int64

### Your turn 3.5
Sites are spelt several ways. Complete the lookup so `Cork Plant` becomes `Cork` and `Limerick Site` becomes `Limerick`.

In [17]:
SITE_MAP = {
    "Cork": "Cork",
    "Cork Plant": "Cork",
    "Galway": "Galway",
    "Limerick": "Limerick",
    "Limerick Site": "Limerick",
}
reg["site"] = reg["site"].str.strip().str.title().map(SITE_MAP)
reg["site"].value_counts(dropna=False)

site
Cork        26
Limerick    19
Galway      17
Name: count, dtype: int64

In [18]:
check(set(reg["site"]) == {"Cork", "Galway", "Limerick"}, "Three sites.", "Some sites did not map.")

PASS: Three sites.


### Units: rated power
Power arrives as `38 kW`, `38kW`, `38000 W` and plain `38`. The `to_kw()` helper converts any of them to kW.
Run and check the result with your engineering eye: is anything not in the range you would expect for these machines?

In [19]:
reg["rated_power_kw"] = reg["rated_power"].apply(to_kw)
reg[["asset_id", "asset_type", "rated_power", "rated_power_kw"]].head(12)

,asset_id,asset_type,rated_power,rated_power_kw
0,CNC-001,CNC Mill,38 kW,38.0
1,CNC-002,CNC Mill,36kW,36.0
2,CNC-003,CNC Mill,38000 W,38.0
3,CNC-004,CNC Mill,28.5,28.5
4,CNC-005,CNC Mill,34.5 kW,34.5
5,CNC-006,CNC Mill,22,22.0
6,CNC-007,CNC Mill,40 KW,40.0
7,CNC-008,CNC Mill,25.5 kW,25.5
8,CNC-009,CNC Mill,38.5kW,38.5
9,CNC-010,CNC Mill,16500 W,16.5


In [20]:
reg["install_date_clean"] = reg["install_date"].apply(parse_install)
reg["install_date_suspect"] = reg["install_date_clean"] > pd.Timestamp.today()
reg.loc[reg["install_date_suspect"], ["asset_id", "install_date", "install_date_clean"]]

,asset_id,install_date,install_date_clean
19,CMP-006,2031-04-12 00:00:00,2031-04-12


An install date in 2031. Probably a typo. We flag it rather than "correct" it.

### Your turn 3.6
Criticality should be A, B or C. Some rows say `High` or `Low`. `High` means `A`, `Low` means `C`.
Blanks become the word `Unassigned`, which makes the gap visible instead of hiding it.

In [21]:
reg["criticality"] = reg["criticality"].replace({"High": "A", "Medium": "B", "Low": "C"})
reg["criticality"] = reg["criticality"].fillna("Unassigned")
reg["criticality"].value_counts()

criticality
B             31
C             17
A             12
Unassigned     2
Name: count, dtype: int64

## 7. Save the clean outputs

In [22]:
wo_clean = wo[["wo_id", "asset_id", "wo_type", "priority", "date_raised", "date_closed", "status",
               "labour_hours", "hours_missing", "parts_cost_eur", "technician", "description", "last_modified"]]
reg_clean = reg.drop(columns=["rated_power", "install_date"]).rename(columns={"install_date_clean": "install_date"})

wo_clean.to_csv("outputs/work_orders_clean.csv", index=False)
reg_clean.to_csv("outputs/asset_register_clean.csv", index=False)
print("Saved", wo_clean.shape, reg_clean.shape)

Saved (1016, 13) (62, 10)


These files disappear when the Colab session ends. Download them from the Files panel (folder icon, left) if you want
them. The next notebook starts from a trainer copy so everyone begins in the same place.

## Recap
* `.str.strip().str.upper()` fixes most text noise.
* A lookup dictionary plus `.map()` fixes spellings, and tells you what it could not map.
* Convert text to numbers, then compare missing values before and after.
* Leave genuinely missing values blank (or labelled). Don't invent them.

Next: **04 Deduplication**.